# 03 - LLM en few-shot et comparaison finale

Ce notebook évalue la troisième approche : un grand modèle de langage (LLM) utilisé **sans
aucun entraînement**. Le modèle reçoit des instructions et quelques exemples annotés directement
dans la requête (*few-shot prompting*), puis classe chaque critique. Il est ensuite comparé aux
deux approches précédentes sur les mêmes critiques :

| Approche | Entraînement | Notebook |
|---|---|---|
| TF-IDF + régression logistique | quelques minutes, CPU | 01 |
| CamemBERT fine-tuné | plusieurs dizaines de minutes, GPU | 02 |
| LLM en few-shot | aucun | 03 |

Deux questions guident l'analyse. La première porte sur la précision : un LLM sans entraînement
rattrape-t-il un modèle spécialisé ? La seconde porte sur le coût : à précision comparable, que
coûtent la latence, le déploiement et l'usage à grande échelle ?

**Deux modes d'utilisation du LLM** sont prévus, au choix via le paramètre `BACKEND` :
- `"anthropic"` : appel d'un modèle Claude via l'API (clé d'API requise, usage facturé) ;
- `"local"` : modèle open source exécuté sur le GPU local (gratuit, mais plus petit).

Comme pour le notebook 02, la première exécution se fait avec `SMOKE_TEST = True` sur un très
petit échantillon, pour vérifier le fonctionnement et estimer le volume de requêtes avant tout
lancement complet.


## 1. Imports et configuration

In [1]:
# Dépendances : pip install anthropic transformers datasets scikit-learn scipy pandas matplotlib tqdm joblib
# (le paquet "anthropic" n'est nécessaire que pour BACKEND = "anthropic")

import getpass
import json
import os
import random
import textwrap
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from scipy.stats import binomtest
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Périphérique pour CamemBERT : {device}")

c:\Users\clement\anaconda3\envs\GPU\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Périphérique pour CamemBERT : cuda


### Paramètres

**Taille de l'échantillon.** Interroger un LLM sur les 20 000 critiques du jeu de test serait
long et, avec une API, coûteux. L'évaluation porte donc sur un échantillon aléatoire fixe
(graine identique à chaque exécution), et les trois approches sont comparées sur **ce même
échantillon**. La conséquence est une marge d'incertitude non négligeable sur chaque
accuracy, quantifiée plus loin par des intervalles de confiance.

**Nombre d'exemples.** `SHOT_COUNTS = [0, 8]` compare un prompt sans exemple (*zero-shot*) à un
prompt avec 8 exemples annotés. Cela mesure l'apport réel des exemples.

**Tarifs.** Le coût en euros ou dollars ne peut être calculé qu'avec les tarifs en vigueur, à
reporter depuis la page de tarification du fournisseur. Ils sont laissés vides : sans eux, le
notebook affiche uniquement le nombre de tokens consommés.

In [2]:
# True : vérification rapide sur un très petit échantillon. False : évaluation complète.
SMOKE_TEST = True

BACKEND = "anthropic"                 # "anthropic" ou "local"
ANTHROPIC_MODEL = "claude-haiku-5-5"  # modèle rapide et économique, adapté à la classification
SHOT_COUNTS = [0, 8]                  # nombres d'exemples comparés dans le prompt

N_EVAL = 50 if SMOKE_TEST else 1000   # critiques évaluées
N_LATENCY = 10 if SMOKE_TEST else 30  # appels séquentiels pour mesurer la latence
MAX_WORKERS = 8                       # requêtes API en parallèle (accélère l'évaluation)

# Tarifs par million de tokens, à renseigner depuis la page de tarification du fournisseur.
PRICE_INPUT_PER_MTOK = None
PRICE_OUTPUT_PER_MTOK = None

if SMOKE_TEST:
    print("MODE TEST RAPIDE : échantillon minuscule, résultats non significatifs.")

MODE TEST RAPIDE : échantillon minuscule, résultats non significatifs.


## 2. Données

L'échantillon d'évaluation provient du jeu de **test**. Les exemples du prompt sont tirés du jeu
d'**entraînement**, jamais du test, ce qui évite toute fuite d'information.

Les exemples sont limités aux critiques courtes (moins de 400 caractères) pour garder un prompt
compact, avec autant de positifs que de négatifs, dans un ordre mélangé. Un ordre régulier
(tous les positifs puis tous les négatifs) peut biaiser un LLM vers le dernier label vu.

In [3]:
ds = load_dataset("tblard/allocine")
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

eval_df = test_df.sample(N_EVAL, random_state=SEED).reset_index(drop=True)
print(f"Échantillon d'évaluation : {len(eval_df)} critiques ({eval_df['label'].mean():.1%} positives)")

# Exemples du prompt : tirés du jeu d'entraînement, courts, équilibrés, ordre mélangé
pool = train_df[train_df["review"].str.len() < 400]
n_per_class = max(SHOT_COUNTS) // 2
shots_df = pd.concat([
    pool[pool["label"] == 1].sample(n_per_class, random_state=SEED),
    pool[pool["label"] == 0].sample(n_per_class, random_state=SEED),
]).sample(frac=1, random_state=SEED)

FEW_SHOTS = [(row.review, int(row.label)) for row in shots_df.itertuples()]
print(f"{len(FEW_SHOTS)} exemples disponibles pour le prompt")

Échantillon d'évaluation : 50 critiques (38.0% positives)
8 exemples disponibles pour le prompt


## 3. Construction du prompt

Le prompt se compose d'une consigne système, puis des exemples présentés comme des échanges
passés (le modèle « voit » des critiques déjà classées), et enfin de la critique à classer.

Deux choix limitent les erreurs de format :
- la réponse attendue tient en **un seul mot** (« positif » ou « négatif »), ce qui rend
  l'analyse triviale et réduit le coût en tokens de sortie ;
- le taux de réponses **non interprétables** est mesuré. Une réponse qui ne correspond à aucun
  des deux labels est comptée comme une erreur, pour ne pas surestimer la précision.

In [4]:
SYSTEM_PROMPT = (
    "Classer la critique de film fournie comme positive ou négative. "
    "Répondre uniquement par un seul mot : positif ou négatif."
)

LABEL_WORDS = {1: "positif", 0: "négatif"}


def build_messages(review, shots):
    """Exemples sous forme d'échanges passés, puis la critique à classer."""
    messages = []
    for text, label in shots:
        messages.append({"role": "user", "content": f"Critique : {text}"})
        messages.append({"role": "assistant", "content": LABEL_WORDS[label]})
    messages.append({"role": "user", "content": f"Critique : {review}"})
    return messages


def parse_label(text):
    """Convertit la réponse du modèle en label (1 positif, 0 négatif), ou None si ambigu."""
    t = text.strip().lower()
    if t.startswith("pos"):
        return 1
    if t.startswith("nég") or t.startswith("neg"):
        return 0
    return None

## 4. Fonctions d'évaluation

### Intervalle de confiance

Une accuracy mesurée sur 1 000 critiques est une estimation, pas une valeur exacte. L'intervalle
de Wilson donne la plage de valeurs plausibles pour l'accuracy réelle (niveau de confiance 95 %).
Il est préférable à l'intervalle gaussien classique lorsque la proportion est proche de 1, ce
qui est le cas ici.

### Test de McNemar

Pour comparer deux classifieurs sur les mêmes critiques, seuls comptent les cas où ils
divergent. Le test de McNemar exact vérifie si l'un a raison plus souvent que l'autre au-delà
de ce que le hasard expliquerait.

In [5]:
def wilson_interval(successes, n, z=1.96):
    """Intervalle de confiance de Wilson à 95 % pour une proportion."""
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half


def score(preds, labels):
    """Accuracy, F1 (classe positive) et taux de réponses interprétables.
    Une prédiction absente (None) est comptée comme une erreur."""
    preds = pd.Series(preds).fillna(-1).astype(int).values
    labels = np.asarray(labels)
    correct = preds == labels
    low, high = wilson_interval(int(correct.sum()), len(correct))
    return {
        "accuracy": correct.mean(),
        "ci_low": low,
        "ci_high": high,
        "f1": f1_score(labels, preds, labels=[1], average="macro"),
        "parse_rate": (preds != -1).mean(),
    }


def mcnemar_exact(correct_a, correct_b):
    """Renvoie (cas où seul A a raison, cas où seul B a raison, p-value)."""
    only_a = int((correct_a & ~correct_b).sum())
    only_b = int((~correct_a & correct_b).sum())
    n_diff = only_a + only_b
    p_value = binomtest(only_a, n_diff, 0.5).pvalue if n_diff > 0 else 1.0
    return only_a, only_b, p_value

## 5. Connexion au LLM

### Mode API (`anthropic`)

La clé d'API est lue dans la variable d'environnement `ANTHROPIC_API_KEY`. À défaut, elle est
saisie à l'invite, sans être affichée ni enregistrée dans le notebook. **Une clé ne doit jamais
être écrite en clair dans un notebook ni dans un dépôt GitHub.**

Un appel de test est effectué dès cette étape : il valide la clé et le nom du modèle avant de
lancer des centaines de requêtes. Il vérifie aussi si le modèle accepte le paramètre de
température (fixé à 0 pour des réponses reproductibles), certains modèles récents ne
l'acceptant pas.

### Mode local (`local`)

Un modèle open source de la famille Qwen2.5 est chargé sur le GPU, avec une taille choisie selon
la mémoire vidéo disponible. La génération est déterministe (sans échantillonnage).

In [6]:
def error_result(err):
    return {"raw": f"ERREUR : {type(err).__name__}", "pred": None,
            "in_tok": 0, "out_tok": 0, "latency_ms": np.nan}


if BACKEND == "anthropic":
    import anthropic

    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Clé d'API Anthropic : ")

    client = anthropic.Anthropic(max_retries=5)  # réessais automatiques (limites de débit, erreurs réseau)

    # Test de connexion : valide clé et modèle, et détecte si la température est acceptée
    USE_TEMPERATURE = True
    probe = dict(model=ANTHROPIC_MODEL, max_tokens=8,
                 messages=[{"role": "user", "content": "Répondre par un seul mot : de quelle couleur est le ciel ?"}])
    try:
        client.messages.create(temperature=0, **probe)
    except anthropic.BadRequestError as err:
        USE_TEMPERATURE = False
        print(f"Température non acceptée par ce modèle, paramètre retiré ({err.message[:80]})")

    def classify(review, shots):
        kwargs = dict(model=ANTHROPIC_MODEL, max_tokens=8, system=SYSTEM_PROMPT,
                      messages=build_messages(review, shots))
        if USE_TEMPERATURE:
            kwargs["temperature"] = 0
        start = time.perf_counter()
        response = client.messages.create(**kwargs)
        latency_ms = (time.perf_counter() - start) * 1000
        text = "".join(block.text for block in response.content if block.type == "text")
        return {"raw": text, "pred": parse_label(text),
                "in_tok": response.usage.input_tokens, "out_tok": response.usage.output_tokens,
                "latency_ms": latency_ms}

    LLM_NAME = ANTHROPIC_MODEL
    LLM_HARDWARE = "API distante"
    print(f"Connexion OK : {ANTHROPIC_MODEL}")

else:
    from transformers import AutoModelForCausalLM

    assert torch.cuda.is_available(), "Le mode local nécessite un GPU."
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    LOCAL_MODEL = (
        "Qwen/Qwen2.5-7B-Instruct" if vram_gb >= 20
        else "Qwen/Qwen2.5-3B-Instruct" if vram_gb >= 8
        else "Qwen/Qwen2.5-1.5B-Instruct"
    )
    llm_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

    llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    try:
        llm = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=llm_dtype).to("cuda").eval()
    except TypeError:  # anciennes versions de transformers : l'argument s'appelait torch_dtype
        llm = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=llm_dtype).to("cuda").eval()

    @torch.no_grad()
    def classify(review, shots):
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + build_messages(review, shots)
        inputs = llm_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to("cuda")
        n_in = inputs["input_ids"].shape[1]

        start = time.perf_counter()
        output = llm.generate(**inputs, max_new_tokens=4, do_sample=False,
                              pad_token_id=llm_tokenizer.eos_token_id)
        torch.cuda.synchronize()
        latency_ms = (time.perf_counter() - start) * 1000

        new_tokens = output[0, n_in:]
        text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)
        return {"raw": text, "pred": parse_label(text), "in_tok": n_in,
                "out_tok": len(new_tokens), "latency_ms": latency_ms}

    LLM_NAME = LOCAL_MODEL
    LLM_HARDWARE = torch.cuda.get_device_properties(0).name
    print(f"Modèle local chargé : {LOCAL_MODEL} ({vram_gb:.0f} Go de VRAM disponibles)")


def safe_classify(review, shots):
    """Une requête en échec ne doit pas interrompre toute l'évaluation."""
    try:
        return classify(review, shots)
    except Exception as err:
        return error_result(err)

ModuleNotFoundError: No module named 'anthropic'

### Vérification sur quelques critiques

Avant de lancer l'évaluation complète, l'inspection des réponses brutes sur trois critiques
permet de s'assurer que le format de sortie est bien celui attendu, et donc que l'analyse
automatique ne rejettera pas de réponses valides.

In [ ]:
for review, label in zip(eval_df["review"].head(3), eval_df["label"].head(3)):
    result = safe_classify(review, FEW_SHOTS)
    print(f"Réel : {LABEL_WORDS[label]:<8} | réponse brute : {result['raw']!r} | interprétée : {result['pred']}"
          f" | {result['in_tok']} tokens en entrée")
    print(textwrap.fill(review[:200], width=100), "\n")

## 6. Évaluation du LLM

Chaque configuration (nombre d'exemples) est évaluée sur l'ensemble de l'échantillon. En mode
API, les requêtes sont envoyées en parallèle pour accélérer l'évaluation. Ce parallélisme rend
les temps de réponse individuels non représentatifs : la latence est donc mesurée à part, en
mode séquentiel (section 7).

In [ ]:
def run_eval(df, shots):
    reviews = df["review"].tolist()
    desc = f"{len(shots)} exemple(s)"
    if BACKEND == "anthropic":
        with ThreadPoolExecutor(MAX_WORKERS) as pool_executor:
            outputs = list(tqdm(pool_executor.map(lambda r: safe_classify(r, shots), reviews),
                                total=len(reviews), desc=desc))
    else:
        outputs = [safe_classify(r, shots) for r in tqdm(reviews, desc=desc)]
    out = pd.DataFrame(outputs)
    out["label"] = df["label"].values
    return out


llm_runs = {}
for k in SHOT_COUNTS:
    llm_runs[k] = run_eval(eval_df, FEW_SHOTS[:k])
    s = score(llm_runs[k]["pred"], llm_runs[k]["label"])
    n_errors = int(llm_runs[k]["raw"].str.startswith("ERREUR").sum())
    print(f"{k} exemple(s) : accuracy {s['accuracy']:.3f} "
          f"[{s['ci_low']:.3f} - {s['ci_high']:.3f}] | réponses interprétables : {s['parse_rate']:.1%}"
          f" | requêtes en échec : {n_errors}")

In [ ]:
# Réponses brutes non interprétées : à inspecter si le taux d'interprétation est inférieur à 100 %
for k, run in llm_runs.items():
    unparsed = run[run["pred"].isna()]
    if len(unparsed):
        print(f"--- {k} exemple(s) : {len(unparsed)} réponse(s) non interprétée(s) ---")
        print(unparsed["raw"].value_counts().head(5).to_string())

## 7. Coût d'usage : latence et tokens

**Latence.** Mesurée sur des appels séquentiels, une critique à la fois, comme dans une
application qui répond à un utilisateur. En mode API, elle inclut le temps réseau.

**Tokens.** Le nombre moyen de tokens consommés par critique détermine la facture. Les exemples
du prompt sont renvoyés à chaque requête : le *few-shot* coûte donc plus cher que le *zero-shot*,
alors que le gain en précision n'est pas garanti.

In [ ]:
latency_sample = eval_df["review"].head(N_LATENCY).tolist()
llm_costs = {}

for k in SHOT_COUNTS:
    shots = FEW_SHOTS[:k]
    safe_classify(latency_sample[0], shots)  # échauffement
    measures = [safe_classify(r, shots)["latency_ms"] for r in latency_sample]

    avg_in = llm_runs[k]["in_tok"].mean()
    avg_out = llm_runs[k]["out_tok"].mean()
    cost_per_1000 = None
    if PRICE_INPUT_PER_MTOK is not None and PRICE_OUTPUT_PER_MTOK is not None:
        cost_per_1000 = (avg_in * PRICE_INPUT_PER_MTOK + avg_out * PRICE_OUTPUT_PER_MTOK) * 1000 / 1e6

    llm_costs[k] = {
        "latency_ms": float(np.nanmedian(measures)),
        "avg_in_tok": float(avg_in),
        "avg_out_tok": float(avg_out),
        "cost_per_1000": cost_per_1000,
    }
    cost_txt = f"{cost_per_1000:.3f} par 1 000 critiques" if cost_per_1000 is not None else "tarifs non renseignés"
    print(f"{k} exemple(s) : latence médiane {llm_costs[k]['latency_ms']:.0f} ms | "
          f"{avg_in:.0f} tokens en entrée, {avg_out:.1f} en sortie | {cost_txt}")

## 8. Les deux autres approches sur le même échantillon

Pour une comparaison équitable, la baseline et CamemBERT sont rechargées depuis leurs
sauvegardes et appliquées à **exactement les mêmes critiques** que le LLM. Cela permet aussi de
comparer leurs erreurs critique par critique.

La baseline provient du notebook 01 et CamemBERT du notebook 02. Si l'une des deux n'est pas
disponible (fichier absent, version de scikit-learn incompatible), elle est simplement omise du
comparatif.

In [ ]:
predictions = {}   # nom de l'approche -> prédictions sur eval_df
meta = {}          # nom de l'approche -> métriques de coût issues des notebooks 01 et 02


def load_json(name):
    path = RESULTS_DIR / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


# --- Baseline TF-IDF + régression logistique ---
try:
    baseline = joblib.load(RESULTS_DIR / "baseline_tfidf_logreg.joblib")
    predictions["TF-IDF + régression logistique"] = baseline.predict(eval_df["review"])
    meta["TF-IDF + régression logistique"] = load_json("baseline.json")
except Exception as err:
    print(f"Baseline non chargée : {type(err).__name__}: {err}")
    print("Pistes : relancer le notebook 01 dans cet environnement pour régénérer le fichier.")

# --- CamemBERT fine-tuné ---
camembert_dir = next((p for p in [Path("models/camembert_allocine"), Path("models/_smoke_test")] if p.exists()), None)

if camembert_dir is None:
    print("CamemBERT non trouvé : exécuter d'abord le notebook 02.")
else:
    if camembert_dir.name == "_smoke_test":
        print("ATTENTION : seul le modèle du test rapide est disponible, ses résultats ne sont pas représentatifs.")

    bert_meta = load_json("camembert.json") or load_json("camembert_smoke_test.json")
    max_length = bert_meta.get("max_length", 256)
    amp_dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16

    bert_tokenizer = AutoTokenizer.from_pretrained(camembert_dir)
    bert_model = AutoModelForSequenceClassification.from_pretrained(camembert_dir).to(device).eval()

    texts, bert_preds = eval_df["review"].tolist(), []
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            batch = bert_tokenizer(texts[i:i + 32], padding=True, truncation=True,
                                   max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                logits = bert_model(**batch).logits
            bert_preds.extend(logits.argmax(-1).cpu().tolist())

    predictions["CamemBERT fine-tuné"] = np.array(bert_preds)
    meta["CamemBERT fine-tuné"] = bert_meta
    del bert_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

for k in SHOT_COUNTS:
    predictions[f"LLM {k} exemple(s)"] = llm_runs[k]["pred"].values

print("\nApproches comparées :", list(predictions))

## 9. Tableau comparatif

Les colonnes de précision sont calculées ici, sur l'échantillon commun. Les colonnes de coût
d'entraînement et de taille proviennent des notebooks précédents. Le « — » signale une valeur
sans objet (par exemple, un LLM utilisé tel quel n'a pas de temps d'entraînement).

In [ ]:
labels = eval_df["label"].values
rows = []

for name, preds in predictions.items():
    s = score(preds, labels)
    m = meta.get(name, {})
    row = {
        "Approche": name,
        "Accuracy": f"{s['accuracy']:.3f}",
        "IC 95 %": f"[{s['ci_low']:.3f} ; {s['ci_high']:.3f}]",
        "F1 (positif)": f"{s['f1']:.3f}",
    }

    if name.startswith("LLM"):
        k = int(name.split()[1])
        c = llm_costs[k]
        row["Entraînement"] = "—"
        row["Latence / critique"] = f"{c['latency_ms']:.0f} ms ({LLM_HARDWARE})"
        row["Taille"] = "—"
        row["Coût / 1 000 critiques"] = (f"{c['cost_per_1000']:.3f}" if c["cost_per_1000"] is not None
                                          else f"{c['avg_in_tok'] * 1000 / 1e6:.2f} M tokens en entrée")
    elif name.startswith("TF-IDF"):
        row["Entraînement"] = f"{m.get('train_time_s', np.nan):.0f} s (CPU)"
        row["Latence / critique"] = f"{m.get('latency_ms', np.nan):.2f} ms (CPU)"
        row["Taille"] = f"{m.get('size_mb', np.nan):.0f} Mo"
        row["Coût / 1 000 critiques"] = "matériel local"
    else:
        row["Entraînement"] = f"{m.get('train_time_s', np.nan) / 60:.0f} min ({m.get('hardware', 'GPU')})"
        row["Latence / critique"] = (f"{m.get('latency_ms_gpu', np.nan):.0f} ms (GPU) / "
                                      f"{m.get('latency_ms_cpu', np.nan):.0f} ms (CPU)")
        row["Taille"] = f"{m.get('size_mb', np.nan):.0f} Mo"
        row["Coût / 1 000 critiques"] = "matériel local"
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

names = list(predictions)
stats = [score(predictions[n], labels) for n in names]
acc = np.array([s["accuracy"] for s in stats])
err_low = acc - np.array([s["ci_low"] for s in stats])
err_high = np.array([s["ci_high"] for s in stats]) - acc

axes[0].barh(names, acc, xerr=[err_low, err_high], color="tab:blue", capsize=4)
axes[0].set_xlabel("Accuracy (barres d'erreur : IC 95 %)")
axes[0].set_xlim(max(0.0, acc.min() - 0.1), 1.0)
axes[0].set_title("Précision sur l'échantillon commun")
axes[0].invert_yaxis()

latencies = []
for n in names:
    if n.startswith("LLM"):
        latencies.append(llm_costs[int(n.split()[1])]["latency_ms"])
    elif n.startswith("TF-IDF"):
        latencies.append(meta[n].get("latency_ms", np.nan))
    else:
        latencies.append(meta[n].get("latency_ms_gpu", np.nan))

axes[1].barh(names, latencies, color="tab:orange")
axes[1].set_xscale("log")
axes[1].set_xlabel("Latence par critique (ms, échelle logarithmique)")
axes[1].set_title("Coût de latence")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. Les différences sont-elles réelles ?

Les intervalles de confiance se chevauchent souvent lorsque les précisions sont proches. Le test
de McNemar compare chaque paire d'approches sur les mêmes critiques. Une p-value inférieure à
0,05 indique que l'écart observé est peu probablement dû au hasard de l'échantillon.

Avec un échantillon de 1 000 critiques, seuls des écarts de plusieurs points sont détectables.
Une absence de significativité ne prouve donc pas que deux approches sont équivalentes : elle
signifie que l'échantillon ne permet pas de trancher.

In [ ]:
correct = {name: np.asarray(pd.Series(p).fillna(-1).astype(int).values == labels) for name, p in predictions.items()}

pairs = []
for a, b in combinations(correct, 2):
    only_a, only_b, p_value = mcnemar_exact(correct[a], correct[b])
    pairs.append({
        "Approche A": a, "Approche B": b,
        "Seule A juste": only_a, "Seule B juste": only_b,
        "p-value": f"{p_value:.3f}",
        "Écart significatif (5 %)": "oui" if p_value < 0.05 else "non",
    })

pd.DataFrame(pairs)

## 11. Critiques mal classées par toutes les approches

Lorsque des modèles très différents (un TF-IDF, un transformer, un LLM) se trompent tous sur la
même critique, la cause est rarement le modèle. Il s'agit plus souvent d'un label discutable,
d'une critique ambiguë ou ironique, ou d'un texte dont la note ne reflète pas le contenu. Ces
cas donnent une idée du plafond de précision réellement atteignable sur ce corpus.

In [ ]:
all_wrong = ~np.any(np.vstack(list(correct.values())), axis=0)
print(f"{all_wrong.sum()} critique(s) sur {len(eval_df)} mal classée(s) par toutes les approches\n")

for _, row in eval_df[all_wrong].head(5).iterrows():
    truth = LABEL_WORDS[row["label"]]
    print(f"[Label du corpus : {truth}]")
    print(textwrap.fill(row["review"][:500], width=100))
    print("-" * 100)

## 12. Sauvegarde

Les résultats du LLM et le tableau comparatif sont enregistrés. En mode test rapide, les
fichiers portent un suffixe distinct pour ne pas être confondus avec les résultats définitifs.

In [ ]:
suffix = "_smoke_test" if SMOKE_TEST else ""

llm_results = []
for k in SHOT_COUNTS:
    s = score(llm_runs[k]["pred"], llm_runs[k]["label"])
    llm_results.append({
        "model": LLM_NAME, "backend": BACKEND, "n_shots": k, "n_eval": N_EVAL,
        "accuracy": round(s["accuracy"], 4), "ci_low": round(s["ci_low"], 4), "ci_high": round(s["ci_high"], 4),
        "f1": round(s["f1"], 4), "parse_rate": round(s["parse_rate"], 4),
        **{key: (round(v, 3) if v is not None else None) for key, v in llm_costs[k].items()},
        "hardware": LLM_HARDWARE, "smoke_test": SMOKE_TEST,
    })

(RESULTS_DIR / f"llm_fewshot{suffix}.json").write_text(
    json.dumps(llm_results, ensure_ascii=False, indent=2), encoding="utf-8")
comparison.to_csv(RESULTS_DIR / f"comparison{suffix}.csv", index=False, encoding="utf-8")

print(f"Résultats enregistrés dans {RESULTS_DIR}/ (suffixe : '{suffix}')")

## Synthèse

À compléter après l'exécution complète (`SMOKE_TEST = False`), en répondant aux questions
suivantes à partir du tableau et des tests ci-dessus :

1. **Précision** : quel est l'ordre des trois approches, et les écarts sont-ils significatifs ?
2. **Apport des exemples** : le prompt avec 8 exemples dépasse-t-il le prompt sans exemple, et
   cet apport justifie-t-il les tokens supplémentaires facturés à chaque requête ?
3. **Coût** : à précision comparable, quelle approche est la plus économique à grande échelle ?
   Une baseline qui répond en une fraction de milliseconde sur CPU change le calcul par rapport
   à un appel d'API facturé à chaque critique.
4. **Interprétabilité** : la baseline expose les mots qui pèsent dans sa décision, ce que ni le
   transformer ni le LLM ne permettent aussi directement.
5. **Plafond** : les critiques mal classées par toutes les approches révèlent-elles des labels
   discutables ?
6. **Contamination possible** : Allociné est un corpus public, présent sur le web depuis des
   années. Un LLM peut en avoir vu des extraits pendant son pré-entraînement, ce qui pourrait
   gonfler artificiellement son score. Cette limite est à mentionner explicitement, et seule
   une évaluation sur des critiques inédites permettrait de l'écarter.

La conclusion attendue n'est pas forcément « le plus gros modèle gagne » : le bon choix dépend
du volume traité, de la latence acceptable, du budget et du besoin d'explicabilité.
